# Google Play Store: What Makes an Android App Succeed?
*Market-entry analysis for a startup launching on Android*

## Overview
**BLUF:** Installs, not ratings, separate successful apps, and installs are concentrated in a few categories. Communication, Video Players, Social and Photography average the most installs per app. Family, Tools and Business are crowded but low-yield. Free apps have a median of 100,000 installs versus 1,000 for paid apps, so a free, ad- or in-app-purchase-supported model is the safer entry strategy. Ratings are high everywhere (mean about 4.2) and barely correlate with installs, so quality is necessary but not a differentiator.

**Story:** (1) where is demand, (2) where is competition, (3) which pricing model works, (4) what app traits matter.

## Business Understanding
**Stakeholders:** founders and product lead of a startup choosing its first Android app and monetisation model.

**Questions**
1. Which categories have the most total installs, and which have the most installs *per app* (demand vs. competition)?
2. Do free or paid apps get more installs and better ratings?
3. How do ratings relate to reviews and installs?
4. Does price point matter for paid apps?
5. Do app size, content rating, or update recency relate to success?
6. Which Install-tier mix does each category have (are there breakout hits or is it flat)?

**Success metric:** Installs (reach). Rating and Reviews are secondary signals.

## Data Understanding
**Source:** `google_play_store_dataset.csv`, a scrape of Google Play listings (10,841 rows, 13 columns, last updates through 2018).

**Known issues found in profiling**
| Issue | Evidence | Decision |
|---|---|---|
| Corrupted row (columns shifted left) | `Life Made WI-Fi Touchscreen Photo Frame`: Category `1.9`, Rating `19.0` | Drop |
| Exact duplicate rows | 483 | Drop |
| Same app listed in several categories | App name repeated with different Category | Keep the row with most reviews |
| `Size` is text (`19M`, `201k`, `Varies with device`) | | Convert to `Size_MB`; "Varies" becomes null |
| `Installs` is text (`10,000+`) | | Strip `+` and `,`, cast to int (lower bound) |
| `Price` is text (`$2.99`) | | Strip `$`, cast to double |
| `Last Updated` is text | | Parse to date, derive year |
| Rating missing for 1,462 apps | mostly low-install apps | Keep rows, do not impute; exclude from rating metrics only |
| Type and Content Rating null (1 each) | | Drop |

**Caveats:** installs are bucketed lower bounds, the data is a 2018 snapshot, and ratings are self-selected.

In [ ]:
# Imports and Spark session
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

spark = (SparkSession.builder
         .appName("PlayStoreAnalysis")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")           # small data: avoid 200 tiny partitions
         .config("spark.sql.legacy.timeParserPolicy", "LEGACY") # tolerant date parsing
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
sns.set_theme(style="whitegrid")

In [ ]:
# Load raw data (multiLine/escape handle quoted commas in app names)
RAW_PATH = "google_play_store_dataset.csv"
try:
    raw = (spark.read.option("header", True).option("multiLine", True)
           .option("escape", '"').csv(RAW_PATH))
except Exception as e:
    raise RuntimeError(f"Could not load {RAW_PATH}: {e}")

print("rows:", raw.count(), "cols:", len(raw.columns))   # expect 10841, 13
raw.printSchema()
raw.show(5, truncate=40)

In [ ]:
# EDA: nulls, duplicates, distinct counts, anomalies
null_counts = raw.select([F.count(F.when(F.col(c).isNull() | (F.col(c) == ""), c)).alias(c) for c in raw.columns])
null_counts.show()

print("exact duplicate rows:", raw.count() - raw.dropDuplicates().count())      # expect 483
print("distinct apps:", raw.select("App").distinct().count())

# anomaly check: Category should be an upper-case label, not a number
raw.filter(~F.col("Category").rlike("^[A-Z_]+$")).show(truncate=30)           # the shifted row

for c in ["Type", "Content Rating", "Installs"]:
    raw.groupBy(c).count().orderBy(F.desc("count")).show(30, truncate=False)

raw.select(F.col("Rating").cast("double").alias("Rating")).describe().show()   # max 19.0 reveals corrupt row

## Data Preparation
Heavy lifting is done in PySpark (type conversion, window-based de-duplication). The result is converted to Pandas for plotting and CSV export, since it is only about 9.7K rows.

In [ ]:
def to_size_mb(col):
    # '19M' -> 19.0 ; '201k' -> 0.196 ; 'Varies with device' -> null
    num = F.regexp_extract(col, r"^([0-9.]+)", 1).cast(DoubleType())
    unit = F.regexp_extract(col, r"([Mk])$", 1)
    return (F.when(unit == "M", num).when(unit == "k", num / 1024).otherwise(F.lit(None)))

def tier(c):
    return (F.when(c < 1e4, "<10K").when(c < 1e5, "10K-100K").when(c < 1e6, "100K-1M")
             .when(c < 1e7, "1M-10M").otherwise("10M+"))

df = (raw
      .filter(F.col("Category") != "1.9")                       # corrupted row
      .dropDuplicates()                                          # exact duplicates
      .withColumn("Reviews", F.col("Reviews").cast("long"))
      .withColumn("Rating", F.col("Rating").cast("double"))
      .withColumn("Size_MB", to_size_mb(F.col("Size")))
      .withColumn("Installs_Num", F.regexp_replace("Installs", "[+,]", "").cast("long"))
      .withColumn("Price_USD", F.regexp_replace("Price", r"\$", "").cast("double"))
      .withColumn("Last_Updated", F.to_date("Last Updated", "MMMM d, yyyy"))
      .withColumn("Update_Year", F.year("Last_Updated"))
      .withColumn("Category", F.initcap(F.regexp_replace("Category", "_", " ")))
      .dropna(subset=["Type", "Content Rating"])
      .withColumn("Install_Tier", tier(F.col("Installs_Num"))))

# One row per app: window function keeps the listing with the most reviews
w = Window.partitionBy("App").orderBy(F.desc("Reviews"), F.desc("Installs_Num"))
clean = (df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
           .drop("Size", "Installs", "Price", "Last Updated")
           .withColumnRenamed("Content Rating", "Content_Rating")
           .withColumnRenamed("Current Ver", "Current_Ver")
           .withColumnRenamed("Android Ver", "Android_Ver")
           .cache())                                             # reused by every analysis cell
print("clean rows:", clean.count())                              # expect 9,658

In [ ]:
# Validation checks (fail loudly if cleaning is wrong)
n = clean.count()
assert clean.select("App").distinct().count() == n,                       "App not unique"
assert clean.filter(F.col("Rating") > 5).count() == 0,                    "Rating out of range"
assert clean.filter(F.col("Installs_Num").isNull()).count() == 0,         "Installs not parsed"
assert clean.filter(F.col("Price_USD").isNull()).count() == 0,            "Price not parsed"
assert clean.filter(F.col("Last_Updated").isNull()).count() == 0,         "Date not parsed"
assert clean.filter((F.col("Type") == "Free") & (F.col("Price_USD") > 0)).count() == 0, "Free app with price"
assert set(r[0] for r in clean.select("Type").distinct().collect()) == {"Free", "Paid"}
print("All validation checks passed. Rating nulls kept:", clean.filter(F.col("Rating").isNull()).count())

In [ ]:
# Save cleaned data for Tableau
pdf = clean.toPandas()
pdf.to_csv("google_play_store_cleaned.csv", index=False)
chk = pd.read_csv("google_play_store_cleaned.csv")
assert chk.shape == pdf.shape
print(chk.shape); chk.head()

**Cleaning outcome:** 10,841 raw rows -> 9,658 apps. Removed 1 corrupted row, 483 exact duplicates, about 697 repeat listings of the same app, and 2 rows with null Type/Content Rating. 1,462 missing ratings are retained (not imputed) so install-based analysis keeps all apps.

## Data Analysis

In [ ]:
# Q1 Demand vs competition by category
cat = (clean.groupBy("Category")
       .agg(F.count("*").alias("apps"), F.sum("Installs_Num").alias("installs"), F.avg("Rating").alias("avg_rating"))
       .withColumn("installs_per_app", F.col("installs") / F.col("apps"))
       .toPandas())
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
cat.nlargest(10, "installs").sort_values("installs").plot.barh(x="Category", y="installs", ax=ax[0], legend=False, title="Total installs (top 10)")
cat.nlargest(10, "installs_per_app").sort_values("installs_per_app").plot.barh(x="Category", y="installs_per_app", ax=ax[1], legend=False, title="Installs per app (top 10)")
plt.tight_layout(); plt.show()

# Bubble: competition (apps) vs installs per app
plt.figure(figsize=(9, 6)); sns.scatterplot(data=cat, x="apps", y="installs_per_app", size="installs", hue="avg_rating", sizes=(40, 600), legend=False)
for _, r in cat.nlargest(8, "installs_per_app").iterrows(): plt.annotate(r.Category, (r.apps, r.installs_per_app), fontsize=8)
plt.title("Competition vs reward by category"); plt.show()

In [ ]:
# Q2 Free vs paid
ft = clean.groupBy("Type").agg(F.count("*").alias("apps"),
        F.expr("percentile_approx(Installs_Num, 0.5)").alias("median_installs"),
        F.round(F.avg("Rating"), 2).alias("avg_rating")).toPandas()
print(ft)
sns.boxplot(data=pdf.assign(log_installs=pdf.Installs_Num.clip(lower=1).apply(lambda x: __import__("math").log10(x))), x="Type", y="log_installs")
plt.title("Installs (log10) by Type"); plt.show()

# Q4 price bands for paid apps
paid = pdf[pdf.Type == "Paid"].copy()
paid["band"] = pd.cut(paid.Price_USD, [0, 1, 3, 5, 10, 1000], labels=["<=$1", "$1-3", "$3-5", "$5-10", ">$10"])
print(paid.groupby("band", observed=True).agg(apps=("App", "count"), median_installs=("Installs_Num", "median")))

In [ ]:
# Q3 and Q5 Relationships (Spearman, because install/review data is heavily skewed)
num = pdf[["Rating", "Reviews", "Installs_Num", "Size_MB", "Price_USD"]]
sns.heatmap(num.corr(method="spearman"), annot=True, fmt=".2f", cmap="RdBu_r", center=0); plt.title("Spearman correlation"); plt.show()

print(pdf.groupby("Content_Rating").Installs_Num.median())
print(pdf.groupby("Update_Year").size())
print(pdf.groupby(pd.cut(pdf.Size_MB, [0, 5, 15, 30, 60, 200])).Rating.mean().round(2))

**Findings (verified against a pandas run of the same logic)**
- **Demand:** Game (13.5B installs), Communication (11.0B) and Tools (8.1B) lead on total installs.
- **Reward per app:** Communication (about 35M installs/app), Video Players (24M), Social (23M), Photography (17M), Productivity (15M). **Competition:** Family (1,875 apps), Game (946) and Tools (829) are the most crowded, and Family returns only about 3.3M per app.
- **Pricing:** Free apps: 8,904, median 100K installs, mean rating 4.17. Paid: 754, median 1K installs, mean rating 4.26. Paid apps are rated slightly higher but reach about 100x fewer users. Among paid apps, $3-5 has the highest median installs (5K).
- **Ratings vs success:** Reviews and Installs correlate at 0.97 (Spearman); Rating vs Installs is only 0.03. Ratings are uniformly high, so they do not predict reach.
- **Size / content rating:** Size is weakly related to installs (0.31). Everyone-rated apps dominate by count (not by median installs).
- **Recency:** 65% of apps were updated in 2018, so recently maintained apps dominate the market.

**Caution:** Installs are lower-bound buckets, so medians are approximate; correlation is not causation (big, popular apps also get more reviews).

### Dashboard plan (Tableau Public, multi-page)
| Page | Visualisations | Why this chart | Interactivity |
|---|---|---|---|
| 1. Market Overview | KPI tiles (apps, total installs, avg rating); horizontal bar of installs by category; treemap of app count | Bars rank accurately; treemap shows share of competition | Category filter, Type filter, click-to-highlight |
| 2. Opportunity | Bubble chart: apps (x) vs installs/app (y), size = installs, colour = rating | Shows crowded vs rewarding categories in one view | Parameter: min apps per category; tooltips |
| 3. Pricing & Monetisation | Box plot of installs by Type; bar of median installs by price band; | Distributions are skewed, so medians and boxes beat means | Price-range slider, Type filter |
| 4. App Traits | Heatmap of Install_Tier by Category; scatter Rating vs Reviews; size band bars | Heatmap shows the install mix per category | Content-rating filter; dashboard action: click a category to filter other pages |

Design: consistent palette (one accent colour for "free/target", neutral grey for the rest), navigation buttons at the top, titles that state the takeaway.

### Link to Published Dashboard
*(paste your Tableau Public URL here)*

### AI Usage Log
| Stage | Prompt (summary) | What AI produced | What I changed / verified |
|---|---|---|---|
| Questions | "Act as a data analyst for an Android startup; propose 6 business questions answerable with these columns" | Question list | Dropped questions the data cannot answer |
| EDA | "Write PySpark null, duplicate, and distinct-count profiling for these columns" | Profiling code | Added anomaly check on Category, which found the corrupted row |
| Cleaning | "Convert Size/Installs/Price text to numeric in PySpark, de-duplicate keeping most reviews" | Casts plus window function | Verified row counts against a pandas run; added asserts |
| Analysis | "Which charts suit skewed install counts?" | Suggested log scale and medians | Used Spearman and medians |
*(Replace with your actual prompts and any edits you made.)*

## Conclusion
1. **Target a high-reward, moderate-competition category** such as Communication, Social, Video Players, Photography or Productivity, and avoid Family and Tools unless the product is clearly differentiated.
2. **Launch free.** Free apps reach about 100x more users at the median; monetise through ads or in-app purchases. If any paid tier, price around $3-5.
3. **Do not over-index on ratings;** they are uniformly about 4.2. Invest in acquisition (reviews and installs move together) and keep the app updated.

**Limitations:** 2018 snapshot, install buckets, no revenue data, ratings missing for about 15% of apps. **Next steps:** add revenue/retention data, model installs with regression, track post-2018 trends.